# Cox Mate — Raid Analysis
Visualizations for Chambers of Xeric data collected by Cox Mate.

## 1. Setup & Data Load

In [29]:
import base64
import os
import re
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from IPython.display import HTML, display
from dotenv import load_dotenv

load_dotenv()

STORE_PATH = Path(os.getenv("STORE_PATH", ""))

# Set this to your RuneLite Boss Kills screenshots directory.
# Falls back to the PHOTOS_DIR env variable if set in .env.
PHOTOS_DIR = Path(os.getenv("PHOTOS_DIR", ""))

try:
    df = (
        pl.read_csv(str(STORE_PATH), separator=",", encoding="utf8")
        .with_columns([
            pl.col("date_completed").cast(pl.Date),
            pl.col("date_processed").cast(pl.Date),
            pl.col("raid_type").cast(pl.Categorical),
        ])
        .sort("date_completed")
    )
    print(f"Loaded {len(df):,} rows from {STORE_PATH}")
    print(f"Date range: {df['date_completed'].min()} → {df['date_completed'].max()}")
    if not PHOTOS_DIR or not PHOTOS_DIR.exists():
        print("⚠️  PHOTOS_DIR not set or not found — screenshots will be skipped in section 7.")
    df.head()
except FileNotFoundError:
    print(f"No data file found at {STORE_PATH}. Run `cox_mate process` first to generate it.")
    raise SystemExit

Loaded 1,048 rows from ~\Desktop\data.csv
Date range: 2023-08-12 → 2026-04-13


In [30]:
df.select("points").describe()

statistic,points
str,f64
"""count""",1048.0
"""null_count""",0.0
"""mean""",53972.587786
"""std""",69894.942664
"""min""",858.0
"""25%""",31424.0
"""50%""",46927.0
"""75%""",63244.0
"""max""",814051.0


## 2. Points Over Time

In [31]:
# Add a row index for rolling average
df_indexed = df.with_row_index("idx")

rolling = (
    df_indexed
    .with_columns(
        pl.col("points").rolling_mean(window_size=10).alias("rolling_avg")
    )
)

fig = px.scatter(
    rolling.to_pandas(),
    x="date_completed",
    y="points",
    color="raid_type",
    color_discrete_map={"cm": "#8b5cf6", "regular": "#3b82f6"},
    opacity=0.55,
    title="Personal Points Per Raid Over Time",
    labels={"date_completed": "Date", "points": "Points", "raid_type": "Raid Type"},
    hover_data=["completion_count"],
)

# Rolling average overlay
fig.add_trace(go.Scatter(
    x=rolling["date_completed"].to_list(),
    y=rolling["rolling_avg"].to_list(),
    mode="lines",
    name="10-raid rolling avg",
    line=dict(color="#f59e0b", width=2),
))

fig.update_layout(height=450, legend_title_text="Raid Type")
fig.show()

## 3. Completions Over Time

In [32]:
weekly = (
    df
    .with_columns(
        pl.col("date_completed").dt.truncate("1w").alias("week")
    )
    .group_by(["week", "raid_type"])
    .agg(pl.len().alias("raids"))
    .sort("week")
)

fig = px.bar(
    weekly.to_pandas(),
    x="week",
    y="raids",
    color="raid_type",
    color_discrete_map={"cm": "#8b5cf6", "regular": "#3b82f6"},
    barmode="stack",
    title="Raids Per Week",
    labels={"week": "Week", "raids": "Raid Count", "raid_type": "Raid Type"},
)
fig.update_layout(height=400)
fig.show()

In [33]:
cumulative = (
    df
    .sort("date_completed")
    .with_row_index("raid_num")
    .with_columns((pl.col("raid_num") + 1).alias("cumulative_raids"))
)

fig = px.line(
    cumulative.to_pandas(),
    x="date_completed",
    y="cumulative_raids",
    title="Cumulative Raids Over Time",
    labels={"date_completed": "Date", "cumulative_raids": "Total Raids"},
)
fig.update_traces(line_color="#3b82f6")
fig.update_layout(height=350)
fig.show()

## 4. Points × Completion Count Interplay

In [34]:
fig = px.scatter(
    df.to_pandas(),
    x="completion_count",
    y="points",
    color="raid_type",
    color_discrete_map={"cm": "#8b5cf6", "regular": "#3b82f6"},
    opacity=0.6,
    trendline="ols",
    title="Points vs Completion Count (does performance improve?)",
    labels={"completion_count": "Completion #", "points": "Points", "raid_type": "Raid Type"},
)
fig.update_layout(height=450)
fig.show()

In [35]:
fig = px.box(
    df.to_pandas(),
    x="raid_type",
    y="points",
    color="raid_type",
    color_discrete_map={"cm": "#8b5cf6", "regular": "#3b82f6"},
    points="outliers",
    title="Points Distribution: CM vs Regular",
    labels={"raid_type": "Raid Type", "points": "Points"},
)
fig.update_layout(height=400, showlegend=False)
fig.show()

## 5. Best Days & Activity Heatmap

In [36]:
top_days = (
    df
    .group_by("date_completed")
    .agg([
        pl.col("points").sum().alias("total_points"),
        pl.len().alias("raids"),
    ])
    .sort("total_points", descending=True)
    .head(15)
    .sort("total_points")
)

fig = px.bar(
    top_days.to_pandas(),
    x="total_points",
    y=top_days["date_completed"].cast(pl.Utf8).to_list(),
    orientation="h",
    text="raids",
    title="Top 15 Days by Total Points",
    labels={"x": "Total Points", "y": "Date"},
    color="total_points",
    color_continuous_scale="Blues",
)
fig.update_traces(texttemplate="%{text} raids", textposition="inside")
fig.update_layout(height=500, coloraxis_showscale=False)
fig.show()

In [37]:
# Parse hour from filename: e.g. "Chambers of Xeric (42) 2024-01-15_18-30-00.png"
def parse_hour(file_name: str) -> int | None:
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{2})-(\d{2})-(\d{2})", file_name)
    return int(m.group(2)) if m else None

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
# Polars dt.weekday() is ISO: Monday=1 … Sunday=7
_WEEKDAY_MAP = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday",
                5: "Friday", 6: "Saturday", 7: "Sunday"}

heatmap_df = (
    df
    .with_columns([
        pl.col("date_completed").dt.weekday().alias("weekday_num"),
        pl.col("file_name").map_elements(parse_hour, return_dtype=pl.Int64).alias("hour"),
    ])
    .with_columns(
        pl.col("weekday_num")
        .map_elements(lambda x: _WEEKDAY_MAP[x], return_dtype=pl.Utf8)
        .alias("day_of_week")
    )
    .drop_nulls("hour")
    .group_by(["day_of_week", "hour"])
    .agg(pl.len().alias("raids"))
    .sort(["hour"])
    .to_pandas()
)

pivot = heatmap_df.pivot(index="day_of_week", columns="hour", values="raids").fillna(0)
pivot = pivot.reindex([d for d in DAY_ORDER if d in pivot.index])

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[f"{h:02d}:00" for h in pivot.columns],
    y=pivot.index.tolist(),
    colorscale="Blues",
    hovertemplate="%{y} %{x}: %{z} raids<extra></extra>",
))
fig.update_layout(
    title="Raid Activity: Day of Week × Hour of Day",
    xaxis_title="Hour (game time)",
    yaxis_title="Day of Week",
    height=380,
)
fig.show()

C:\Users\matth\AppData\Local\Temp\ipykernel_37404\207054401.py:19: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("weekday_num").map_elements(lambda x: ...)
with this one instead:
  + pl.col("weekday_num").replace_strict(_WEEKDAY_MAP)

  .map_elements(lambda x: _WEEKDAY_MAP[x], return_dtype=pl.Utf8)


## 6. Cumulative Points Over Time

In [ ]:
cum_points = (
    df
    .sort("date_completed")
    .with_columns(
        pl.col("points").cum_sum().alias("cumulative_points")
    )
)

# Split by raid type for stacked area
cum_by_type = (
    df
    .sort("date_completed")
    .with_columns(
        pl.col("points").cum_sum().over("raid_type").alias("cumulative_points")
    )
)

fig = px.area(
    cum_by_type.to_pandas(),
    x="date_completed",
    y="cumulative_points",
    color="raid_type",
    color_discrete_map={"cm": "#8b5cf6", "regular": "#3b82f6"},
    title="Cumulative Points Over Time by Raid Type",
    labels={"date_completed": "Date", "cumulative_points": "Cumulative Points", "raid_type": "Raid Type"},
)
fig.update_layout(height=400)
fig.show()

## 7. Highlights & Lowlights

Top 5 highest and lowest point raids, with screenshots.
Requires `PHOTOS_DIR` to be set in the setup cell or in `.env`.

In [ ]:
def _raid_card(row: dict, photos_dir: Path) -> str:
    """Build an HTML card for a single raid row."""
    img_path = photos_dir / row["file_name"] if (photos_dir and photos_dir.exists()) else None

    if img_path and img_path.exists():
        with open(img_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        img_tag = (
            f"<img src='data:image/png;base64,{b64}' "
            "style='width:100%;max-width:480px;border-radius:6px;display:block'>"
        )
    else:
        img_tag = "<div style='width:480px;height:270px;background:#1e1e2e;border-radius:6px;display:flex;align-items:center;justify-content:center;color:#666'>screenshot not found</div>"

    label = (
        f"<div style='margin-top:6px;font-size:13px;color:#cdd6f4'>"
        f"<b style='font-size:16px;color:#cba6f7'>{row['points']:,} pts</b> &nbsp;·&nbsp; "
        f"{row['raid_type'].upper()} &nbsp;·&nbsp; "
        f"completion #{row['completion_count']:,} &nbsp;·&nbsp; "
        f"{row['date_completed']}"
        f"</div>"
    )

    return (
        "<div style='display:inline-block;vertical-align:top;margin:8px;"
        "background:#181825;padding:10px;border-radius:8px;max-width:500px'>"
        f"{img_tag}{label}</div>"
    )


def show_raid_cards(raids_df: pl.DataFrame, title: str, emoji: str) -> None:
    header = (
        f"<h3 style='font-family:sans-serif;color:#cdd6f4;margin-bottom:4px'>"
        f"{emoji} {title}</h3>"
    )
    cards = "".join(_raid_card(row, PHOTOS_DIR) for row in raids_df.iter_rows(named=True))
    display(HTML(
        f"<div style='background:#11111b;padding:12px 8px;border-radius:10px;margin-bottom:20px'>"
        f"{header}{cards}</div>"
    ))

In [ ]:
TOP_N = 5

highlights = df.sort("points", descending=True).head(TOP_N)
lowlights  = df.sort("points", descending=False).head(TOP_N)

show_raid_cards(highlights, f"Top {TOP_N} Highest Point Raids", "🏆")
show_raid_cards(lowlights,  f"Top {TOP_N} Lowest Point Raids",  "💀")